# Phase Space Analysis

This notebook mirrors `phase_space_analysis.py` and runs full nonlinear phase-space analysis for a one-column CSV time series.

## 1) Imports and configuration

Set the CSV path (default points to `data/diff_sums.csv`) and output directory.

In [ ]:
from phase_space_analysis import (
    AnalysisResults,
    average_mutual_information,
    choose_embedding_dimension,
    delay_embedding,
    false_nearest_neighbors,
    find_poincare_crossings,
    first_local_minimum,
    load_series,
    print_summary,
    rqa_metrics,
    rosenstein_lle,
    save_ami_plot,
    save_fnn_plot,
    save_lyapunov_plot,
    save_phase_portraits,
    save_poincare_plot,
    save_recurrence_plot,
    estimate_mean_period,
)
from scipy.spatial.distance import pdist, squareform
import numpy as np
import os

csv_path = "data/diff_sums.csv"
output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

## 2) Automated embedding parameter estimation

Compute AMI (lags 1..50) to select \(\tau\), then compute FNN (dimensions 1..10) to select \(m\).

In [ ]:
series = load_series(csv_path)
ami_values = average_mutual_information(series, max_lag=50)
tau = first_local_minimum(ami_values)
save_ami_plot(ami_values, tau, output_dir)

dimensions, fnn_percentages = false_nearest_neighbors(series, tau=tau, max_dim=10, rtol=15.0, atol=2.0)
m = choose_embedding_dimension(dimensions, fnn_percentages, threshold=5.0)
save_fnn_plot(dimensions, fnn_percentages, m, output_dir)

tau, m

## 3) Attractor reconstruction and Poincaré section

Build delay embedding, save 2D/3D attractor plots, and compute mean-crossing positive-slope Poincaré section.

In [ ]:
embedding3d = delay_embedding(series, m=max(3, m), tau=tau)[:, :3]
save_phase_portraits(embedding3d, output_dir)

poincare_points = find_poincare_crossings(embedding3d, threshold=float(np.mean(series)))
save_poincare_plot(poincare_points, output_dir)

poincare_points.shape

## 4) Largest Lyapunov exponent (Rosenstein)

Estimate LLE from nearest-neighbor divergence in the reconstructed phase space.

In [ ]:
embedding_m = delay_embedding(series, m=m, tau=tau)
mean_period = estimate_mean_period(series)
lle, t, divergence, fit_line = rosenstein_lle(embedding_m, mean_period=mean_period, max_t=50)
save_lyapunov_plot(t, divergence, fit_line, output_dir)

lle

## 5) Recurrence plot and RQA metrics

Build recurrence matrix with \epsilon at 10% of max pairwise distance, save recurrence plot, and compute RQA statistics.

In [ ]:
distances = squareform(pdist(embedding_m))
epsilon = 0.1 * np.max(distances)
recurrence = (distances <= epsilon).astype(int)
save_recurrence_plot(recurrence, output_dir)
metrics = rqa_metrics(recurrence, min_line=2)
metrics

## 6) Final summary

Collect and print the same summary as the command-line script.

In [ ]:
results = AnalysisResults(tau=tau, embedding_dimension=m, lle=lle, rqa_metrics=metrics)
print_summary(results)